In [ ]:
from pathlib import Path
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.

제3조 (휴가)
1. 연차휴가는 근로기준법에 따라 부여한다.
2. 경조사 휴가는 별도 규정에 따른다.
3. 자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

1. 개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

2. 주요 트렌드
- RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
- 멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
- AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
- 소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

3. 시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

1. 제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

2. 초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

3. 주요 기능
- 음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
- 자동 스케줄: 시간대별 기기 자동 제어
- 에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
- 보안 모드: 외출 시 자동 보안 설정
"""
}

SAMPLE_DIR = Path('sample_data')
SAMPLE_DIR.mkdir(exist_ok=True)

for filename, content in sample_texts.items():
    (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')

In [ ]:
import os
import time
import numpy as np
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS

llm = ChatOpenAI(model='gpt-4o-mini')
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
from pathlib import Path

data_dir = Path("sample_data")

files = {
    data_dir / "company_policy.txt" : '사내규정',
    data_dir / 'product_manual.txt' : '제품매뉴얼',
    data_dir / 'ai_report.txt' : 'AI보고서'
}

documents = []
for fpath, category in files.items():
    text = fpath.read_text()
    for section in text.strip().split('\n\n'):
        if section.strip():
            documents.append(Document(
                page_content = section.strip(),
                metadata = {'category' : category, 'source' : fpath.name}
            ))

In [4]:
len(documents)

13

In [5]:
documents[0]

Document(metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='주식회사 모두의연구소 사내 규정')

In [6]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

In [8]:
vectorstore.save_local('faiss_docs')

In [9]:
loaded_vs = FAISS.load_local('faiss_docs', embeddings, allow_dangerous_deserialization=True)

In [10]:
loaded_vs.index.ntotal

13

In [11]:
retriever.invoke('재택근무 규정')

[Document(id='e07d13f8-db99-4d50-9114-dd9e7e8cbe63', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.'),
 Document(id='4c699a5a-4ba8-4959-941c-779d4350e492', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.'),
 Document(id='83ec3a3c-8155-4cc7-bd03-6063e783d00d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.')]

In [ ]:
# LCEL : LangChain Expression Language
#      | |  |

In [ ]:
# prompt | llm  | StrOutputParser()

In [12]:
simple_prompt = ChatPromptTemplate.from_messages([
    ('user', '{question}')
])

simple_chain = simple_prompt | llm  | StrOutputParser()

In [13]:
result = simple_chain.invoke({'question': '대한민국의 수도는?'})

In [14]:
print(result)

대한민국의 수도는 서울입니다.


In [ ]:
# query -> retriever 검색을 해올거고   ->  llm 

# llm  : ('재택근무 규정이 어떻게되나요?', 'retrieved documents')
    
# rag_chain = "query" | retriever | "결과" -> |  llm  | StrOutputParser()

In [15]:
# RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()

In [16]:
passthrough.invoke('안녕하세요')

'안녕하세요'

In [17]:
passthrough.invoke(123)

123

In [18]:
passthrough.invoke({'a' : 1})

{'a': 1}

In [19]:
def format_docs(docs):
    return '\n--\n'.join(
            f"[{d.metadata.get('category', '')}] {d.page_content}"
            for d in docs
    )

In [20]:
docs = retriever.invoke('재택근무')

In [21]:
docs

[Document(id='e07d13f8-db99-4d50-9114-dd9e7e8cbe63', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.'),
 Document(id='6d407b1d-11c6-451d-bd88-9f6ce7a035a9', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제4조 (교육)\n1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n3. 온라인 학습 플랫폼 이용료를 전액 지원한다.'),
 Document(id='83ec3a3c-8155-4cc7-bd03-6063e783d00d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.')]

In [22]:
formatted = format_docs(docs)

In [23]:
print(formatted)

[사내규정] 제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.
--
[사내규정] 제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
--
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


In [24]:
retriever_chain = retriever | format_docs
context_text = retriever_chain.invoke('재택근무')

In [25]:
print(context_text)

[사내규정] 제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.
--
[사내규정] 제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
--
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


In [26]:
question = '재택근무 규정이 어떻게 되나요?'
docs = voctorstore.similarity_search(question, k=3)
context = '\n--\n'.join(
            f"[{d.metadata.get('category', '')}] {d.page_content}"
            for d in docs)
messages = [
    SystemMessage(),
    HumanMessage(),
    ...
]

NameError: name 'voctorstore' is not defined

In [27]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system" , "사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요."),
    ("user", "{question}")
])

rag_chain = (  {"context" : retriever | format_docs, 
                "question" : RunnablePassthrough()}
            | rag_prompt
            | llm
            | StrOutputParser()
            )

In [28]:
# rag_chain을 만들어보세요. K=1 새 체인을 만드셔서 비교를 해보세요

In [29]:
retriever_k1 = vectorstore.as_retriever(search_kwargs={'k':1})
rag_chain = (  {"context" : retriever_k1 | format_docs, 
                "question" : RunnablePassthrough()}
            | rag_prompt
            | llm
            | StrOutputParser()
            )

In [30]:
answer = rag_chain.invoke('재택근무 규정이 어떻게 되나요?')
print(answer)

재택근무는 주 2회까지 가능하다는 규정이 있습니다.


In [31]:
question_list = ['스마트홈 허브 초기 설정 방법', 'ai 산업 성장률', '교육비 지원 한도']
for q in question_list:
    print(f"Q : {q}")
    print(f"A : {rag_chain.invoke(q)}")
    print("======")

Q : 스마트홈 허브 초기 설정 방법
A : 스마트 홈 허브 v3.0의 초기 설정 방법은 다음과 같습니다:

1. **전원 연결**:
   - 스마트 홈 허브를 전원 어댑터에 연결한 후, 전원을 켭니다.

2. **Wi-Fi 연결**:
   - 허브의 LED가 깜박이면, 허브의 설정 모드에 들어가게 됩니다.
   - 스마트폰이나 태블릿의 Wi-Fi 설정에서 ‘SmartHub-XXXX’를 선택합니다.
   - 이후 스마트 홈 허브의 전용 앱을 다운로드하고 설치합니다.

3. **앱 등록 및 로그인**:
   - 설치한 앱을 실행하고, 사용자 계정을 생성하거나 기존 계정으로 로그인합니다.

4. **허브 연결 설정**:
   - 앱 화면에서 ‘허브 추가’ 또는 ‘새 장치 추가’를 선택합니다.
   - Wi-Fi 네트워크를 선택하고 비밀번호를 입력하여 허브를 인터넷에 연결합니다.

5. **장치 추가**:
   - 허브 연결이 완료되면, 앱에서 ‘장치 추가’ 기능을 사용하여 다양한 스마트 기기를 연결합니다.

6. **설정 완료**:
   - 모든 장치가 연결되면, 원하는 설정과 자동화를 추가하여 스마트 홈 환경을 설정할 수 있습니다.

스마트 홈 허브의 초기 설정 과정은 간단하며, 설정 중 문제가 발생하면 사용자 매뉴얼의 문제 해결 섹션을 참고하시면 도움이 됩니다.
Q : ai 산업 성장률
A : 2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했습니다. 특히 생성형 AI 분야가 전체 성장의 60%를 차지하고 있습니다.
Q : 교육비 지원 한도
A : 사내규정에 따르면 외부 컨퍼런스 참석비는 연 200만원까지 지원됩니다. 또한, 온라인 학습 플랫폼 이용료는 전액 지원됩니다.


In [32]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system" , "사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요."
                "참고 문서에 없는 내용이면, '해당 정보가 없습니다' 라고 답변하세요."),
    ("user", "{question}")
])

rag_chain = (  {"context" : retriever | format_docs, 
                "question" : RunnablePassthrough()}
            | rag_prompt
            | llm
            | StrOutputParser()
            )


rag_chain.invoke('재택근무 규정이 어떻게 되나요?')

'재택근무는 주  2회까지 가능하다고 규정되어 있습니다.'

In [33]:
from langchain_core.runnables import RunnableParallel

In [34]:
rag_chain_with_sources = RunnableParallel(
                            answer = rag_chain,
                            source_document = retriever)

In [35]:
rag_chain_with_sources.invoke('재택근무 규정이 어떻게 되나요?')

{'answer': '재택근무는 주 2회까지 가능하다고 규정되어 있습니다.',
 'source_document': [Document(id='e07d13f8-db99-4d50-9114-dd9e7e8cbe63', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.'),
  Document(id='4c699a5a-4ba8-4959-941c-779d4350e492', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.'),
  Document(id='83ec3a3c-8155-4cc7-bd03-6063e783d00d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.')]}

In [36]:
result = rag_chain_with_sources.invoke('재택근무 규정이 어떻게 되나요?')
for doc in result['source_document']:
    print(f" [{doc.metadata['category']}] {doc.page_content[:100]}")

 [사내규정] 제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까
 [사내규정] 제3조 (휴가)
1. 연차휴가는 근로기준법에 따라 부여한다.
2. 경조사 휴가는 별도 규정에 따른다.
3. 자기개발 휴가를 연 5일 추가 부여한다.
 [사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


In [ ]:
# 답변을 streaming 으로 한번 받아보세요

In [37]:
for chunk in rag_chain.stream('재택근무 규정이 어떻게 되나요?'):
    print(chunk, end="", flush=True)
    

재택근무는 주  2회까지 가능하다고 규정되어 있습니다.

In [38]:
rag_chain.invoke('2002년 월드컵 우승팀은 어디인가요?')

'해당 정보가 없습니다.'

In [39]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

In [41]:
def check_and_prepare(question):
    results = vectorstore.similarity_search_with_score(question, k=3)
    docs = [doc for doc, _ in results]
    return {
        'question' : question,
        'context' : format_docs(docs),
        'score' : results[0][1]
    }

safe_chain = (
    RunnableLambda(check_and_prepare) 
    | RunnableBranch(
        (lambda x: x['score'] > 1.5 , lambda x : '해당 정보가 없습니다'), 
        rag_prompt | llm | StrOutputParser()
    )
)

safe_chain.invoke('2002년 월드컵 우승팀은 어디인가요?')

'해당 정보가 없습니다'

In [42]:
safe_chain.invoke('재택근무 규정을 알려주세요?')

'재택근무는 주  2회까지 가능하다고 규정되어 있습니다.'

In [43]:
# RunnableParallel: answer, source_documents, 검색된 텍스트 context를 동시에 리턴하는 체인

In [44]:
new_chain = RunnableParallel(
        answer = rag_chain,
        source_documents = retriever,
        context = retriever | format_docs
)
result = new_chain.invoke('재택근무 규정을 알려주세요')

In [45]:
result

{'answer': '재택근무는 주 2회까지 가능하다는 규정이 있습니다.',
 'source_documents': [Document(id='e07d13f8-db99-4d50-9114-dd9e7e8cbe63', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.'),
  Document(id='4c699a5a-4ba8-4959-941c-779d4350e492', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.'),
  Document(id='83ec3a3c-8155-4cc7-bd03-6063e783d00d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.')],
 'context': '[사내규정] 제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.\n--\n[사내규정] 제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.\n--\n[사내규정] 제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임

In [46]:
retriever.invoke('2002년 월드컵')

[Document(id='d4d637d4-ba6b-47ef-95f0-02d79ae100b3', metadata={'category': 'AI보고서', 'source': 'ai_report.txt'}, page_content='1. 개요\n2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.\n특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.'),
 Document(id='baff5a27-0c22-42a5-a975-3e79c6382e3f', metadata={'category': 'AI보고서', 'source': 'ai_report.txt'}, page_content='2024년 인공지능 산업 동향 보고서'),
 Document(id='b3c40d6a-ed34-40bf-9197-471bd894887f', metadata={'category': 'AI보고서', 'source': 'ai_report.txt'}, page_content='3. 시장 전망\n2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.\n특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.')]

In [47]:
vectorstore.similarity_search_with_score('재택근무 규정', k=3)

[(Document(id='e07d13f8-db99-4d50-9114-dd9e7e8cbe63', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.'),
  np.float32(0.99679273)),
 (Document(id='4c699a5a-4ba8-4959-941c-779d4350e492', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.'),
  np.float32(1.241905)),
 (Document(id='83ec3a3c-8155-4cc7-bd03-6063e783d00d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.'),
  np.float32(1.2633799))]

In [48]:
for doc, score in vectorstore.similarity_search_with_score('재택근무 규정', k=3):
    print(f" [{doc.metadata['category']}] {doc.page_content}")

 [사내규정] 제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.
 [사내규정] 제3조 (휴가)
1. 연차휴가는 근로기준법에 따라 부여한다.
2. 경조사 휴가는 별도 규정에 따른다.
3. 자기개발 휴가를 연 5일 추가 부여한다.
 [사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


In [49]:
test_queries = [
    {"query" : "재택근무 몇 회?"},
    {"query" : "스마트홈 초기 설정"},
    {"query" : "RAG 기술 동향"},
]

for tq in test_queries:
    results = vectorstore.similarity_search_with_score(tq['query'], k=1)
#     print(results[0][0].page_content)
    top_cat = results[0][0].metadata['category']
    context = results[0][0].page_content
    top_score = results[0][1]
    print(f"{tq} : top category : {top_cat}, {top_score}, {context}")

{'query': '재택근무 몇 회?'} : top category : 사내규정, 0.9943448305130005, 제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.
{'query': '스마트홈 초기 설정'} : top category : 제품매뉴얼, 0.8595510721206665, 스마트 홈 허브 v3.0 사용자 매뉴얼
{'query': 'RAG 기술 동향'} : top category : AI보고서, 1.132086157798767, 2024년 인공지능 산업 동향 보고서


In [51]:
# vectorstore.similarity_search_with_score
# query 를 입력 -> threshold 설정 -> 답변할 수 없다. / 해당 문서 내용
def verify_query(query, threshold=1.3, k=3):
    results = vectorstore.similarity_search_with_score(query, k=k)
    
#     [(doc, score), (doc, score), (doc, score)]
    best_doc = None
    bset_score = float('inf') #1000000000
    
    for doc, score in results:
        if score < bset_score:
            bset_score = score
            best_doc = doc
    
    if bset_score > threshold:
        return '해당 정보가 없습니다'
    
    return best_doc.page_content

In [52]:
verify_query('2002년 월드컵')

'해당 정보가 없습니다'